# LAB 08 — Pretrained Transformer: Fine-tuning experiments

Chạy trên **Google Colab (GPU)**: *Runtime → Change runtime type → T4 GPU*, rồi *Runtime → Run all*.

Code chính nằm trong `pretrained_classification.py`. Notebook này chạy:

| Phần | Nội dung |
|---|---|
| Main | DistilBERT fine-tuning trên IMDb (100% train, T=128) |
| Exp 1 | Pretrained vs Transformer from scratch |
| Exp 2 | Frozen encoder (feature extraction) vs fine-tuning |
| Exp 3 | 10% / 25% / 50% / 100% training data |
| Exp 4 | Max sequence length 64 / 128 / 256 |
| Exp 5 | Domain shift: IMDb → TweetEval |
| 19–21 | Metrics, confusion matrix, error & confidence analysis |

In [1]:
# Setup: on Google Colab (if the code is not already here), clone the repo and move into lab08/
import os, sys
if "google.colab" in sys.modules and not os.path.exists("pretrained_classification.py"):
    if not os.path.exists("/content/NLPandApp"):
        !git clone -q https://github.com/DuanShine/NLPandApp.git /content/NLPandApp
    %cd /content/NLPandApp/lab08
    !pip -q install -U datasets
print(os.getcwd())

/content/lab08


In [2]:
import time, json
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, logging as hf_logging
hf_logging.set_verbosity_error()

import pretrained_classification as pc
from pretrained_classification import Config, run, evaluate_on

cfg = Config()
pc.set_seed(cfg.seed)
print("Device  :", pc.DEVICE)
print("Hardware:", pc.hardware_info())
print("torch", torch.__version__)
print(cfg)
T_START = time.perf_counter()

Device  : cuda
Hardware: x86_64, 2 logical cores, 13 GB RAM, Tesla T4
torch 2.14.1+cu130
Config(model_name='distilbert-base-uncased', seed=42, n_train=4000, n_val=1000, n_test=2000, n_ood=1000, max_len=128, batch_size=32, ft_epochs=2, ft_lr=2e-05, frozen_epochs=30, frozen_lr=0.001, scratch_epochs=5, scratch_lr=0.0005, weight_decay=0.01, warmup_ratio=0.1)


## Data

In [3]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
data = pc.load_imdb(cfg)
for k, (x, y) in data.items():
    print(f"{k:5s} n={len(x):5d}  positive share={np.mean(y):.3f}")

lens = np.array([len(tokenizer(t)["input_ids"]) for t in data["test"][0]])
print("\nIMDb test review length in tokens: median=%d, mean=%.0f" % (np.median(lens), lens.mean()))
for L in (64, 128, 256, 512):
    print(f"  reviews longer than {L:3d} tokens (get truncated): {np.mean(lens > L):.1%}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (3 outputs left out)

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

train n= 4000  positive share=0.496
val   n= 1000  positive share=0.521
test  n= 2000  positive share=0.500



IMDb test review length in tokens: median=227, mean=296
  reviews longer than  64 tokens (get truncated): 98.1%
  reviews longer than 128 tokens (get truncated): 88.3%
  reviews longer than 256 tokens (get truncated): 41.5%
  reviews longer than 512 tokens (get truncated): 12.8%


## Main run — DistilBERT fine-tuning (100% data, T=128)

In [4]:
rows = []
main = run("pretrained", "fine-tuning", data, tokenizer, cfg, "main")
rows.append(main.row)
y_test = np.array(data["test"][1])
pred_main = main.probs.argmax(1)
print(classification_report(y_test, pred_main, target_names=pc.LABELS, digits=4))
fig, cm = pc.plot_confusion(y_test, pred_main, "confusion_matrix.png")
plt.show()
print("confusion matrix [[TN, FP], [FN, TP]] =", cm.tolist())

[main] pretrained/fine-tuning  train=4000  max_len=128


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  epoch 1/2  loss=0.4927  val_acc=0.8470  (43s)


  epoch 2/2  loss=0.2893  val_acc=0.8430  (88s)


  -> acc=0.8440 f1=0.8468 time=88.1s
              precision    recall  f1-score   support

    negative     0.8568    0.8260    0.8411      1000
    positive     0.8320    0.8620    0.8468      1000

    accuracy                         0.8440      2000
   macro avg     0.8444    0.8440    0.8439      2000
weighted avg     0.8444    0.8440    0.8439      2000

confusion matrix [[TN, FP], [FN, TP]] = [[826, 174], [138, 862]]


## Experiment 2 — Frozen encoder vs fine-tuning

In [5]:
frozen = run("pretrained", "frozen", data, tokenizer, cfg, "exp2_frozen_vs_ft")
rows.append(frozen.row)
cols = ["mode", "encoder_trainable", "accuracy", "f1", "train_time_s", "params_trainable"]
pd.DataFrame([main.row, frozen.row])[cols]

[exp2_frozen_vs_ft] pretrained/frozen  train=4000  max_len=128


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  frozen head: final loss=0.4007 val_acc=0.7770


  -> acc=0.8000 f1=0.7957 time=20.4s


,mode,encoder_trainable,accuracy,f1,train_time_s,params_trainable
0,fine-tuning,True,0.844,0.846758,88.1,66364418
1,frozen,False,0.800,0.795710,20.4,1538


## Experiment 1 — Pretrained vs Transformer from scratch

In [6]:
scratch = run("scratch", "from scratch", data, tokenizer, cfg, "exp1_scratch_vs_pretrained")
rows.append(scratch.row)
cols = ["model", "epochs", "accuracy", "precision", "recall", "f1", "train_time_s", "params_total"]
pd.DataFrame([scratch.row, main.row])[cols]

[exp1_scratch_vs_pretrained] scratch/from scratch  train=4000  max_len=128


  epoch 1/5  loss=0.6923  val_acc=0.5390  (2s)


  epoch 2/5  loss=0.6222  val_acc=0.6690  (3s)


  epoch 3/5  loss=0.5168  val_acc=0.6860  (4s)


  epoch 4/5  loss=0.4284  val_acc=0.7020  (6s)


  epoch 5/5  loss=0.3766  val_acc=0.6970  (7s)
  -> acc=0.6875 f1=0.6858 time=7.0s


,model,epochs,accuracy,precision,recall,f1,train_time_s,params_total
0,Transformer from scratch,5,0.6875,0.689585,0.682,0.685772,7.0,4188674
1,DistilBERT (pretrained),2,0.8440,0.832046,0.862,0.846758,88.1,66364418


## Experiment 3 — Amount of training data (validation/test fixed)

In [7]:
eff_rows = [main.row, scratch.row]
for frac in (0.1, 0.25, 0.5):
    for kind, mode in (("pretrained", "fine-tuning"), ("scratch", "from scratch")):
        r = run(kind, mode, data, tokenizer, cfg, "exp3_data", train_frac=frac)
        rows.append(r.row)
        eff_rows.append(r.row)
eff = pd.DataFrame(eff_rows)
pc.plot_data_efficiency(eff, "data_efficiency.png")
plt.show()
eff.pivot_table(index="train_frac", columns="model", values=["accuracy", "f1"]).round(4)

[exp3_data] pretrained/fine-tuning  train=400  max_len=128


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  epoch 1/2  loss=0.6834  val_acc=0.5850  (8s)


  epoch 2/2  loss=0.6251  val_acc=0.6990  (15s)


  -> acc=0.6995 f1=0.7227 time=15.1s


[exp3_data] scratch/from scratch  train=400  max_len=128


... (22 outputs left out)

  epoch 3/5  loss=0.5801  val_acc=0.6290  (2s)


  epoch 4/5  loss=0.5180  val_acc=0.6750  (3s)


  epoch 5/5  loss=0.4551  val_acc=0.6810  (4s)
  -> acc=0.6465 f1=0.6563 time=3.5s


accuracy                           \
model      DistilBERT (pretrained) Transformer from scratch   
train_frac                                                    
0.10                        0.6995                   0.5880   
0.25                        0.8050                   0.6265   
0.50                        0.8430                   0.6465   
1.00                        0.8440                   0.6875   

                                f1                           
model      DistilBERT (pretrained) Transformer from scratch  
train_frac                                                   
0.10                        0.7227                   0.5774  
0.25                        0.8081                   0.6274  
0.50                        0.8456                   0.6563  
1.00                        0.8468                   0.6858

## Experiment 4 — Maximum sequence length

In [8]:
len_rows = [main.row]
for L in (64, 256):
    r = run("pretrained", "fine-tuning", data, tokenizer, cfg, "exp4_maxlen", max_len=L)
    rows.append(r.row)
    len_rows.append(r.row)
cols = ["max_len", "accuracy", "f1", "train_time_s", "peak_mem_mb"]
pd.DataFrame(len_rows)[cols].sort_values("max_len")

[exp4_maxlen] pretrained/fine-tuning  train=4000  max_len=64


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  epoch 1/2  loss=0.5506  val_acc=0.7810  (23s)


  epoch 2/2  loss=0.3687  val_acc=0.7910  (47s)


  -> acc=0.7920 f1=0.7992 time=47.3s


[exp4_maxlen] pretrained/fine-tuning  train=4000  max_len=256


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  epoch 1/2  loss=0.4406  val_acc=0.8690  (91s)


  epoch 2/2  loss=0.2299  val_acc=0.8890  (182s)


  -> acc=0.8905 f1=0.8937 time=181.7s


,max_len,accuracy,f1,train_time_s,peak_mem_mb
1,64,0.7920,0.799228,47.3,2263
0,128,0.8440,0.846758,88.1,2084
2,256,0.8905,0.893741,181.7,4665


## Experiment 5 — Domain shift (train: IMDb movie reviews → test: TweetEval tweets)

In [9]:
tw_texts, tw_labels = pc.load_tweets(cfg)
print("OOD set:", len(tw_texts), "tweets, positive share = %.3f" % np.mean(tw_labels))
ood = {}
shift = []
for name, r in (("fine-tuned", main), ("frozen", frozen), ("from scratch", scratch)):
    m_ood, p_ood = evaluate_on(r.model, tokenizer, tw_texts, tw_labels, cfg.max_len)
    ood[name] = p_ood
    rows.append({**r.row, "experiment": "exp5_domain_shift", "eval_set": "TweetEval (OOD)", **m_ood})
    shift.append({"model": f"{r.row['model']} / {r.row['mode']}",
                  "acc in-domain (IMDb)": r.row["accuracy"], "acc out-of-domain (tweets)": m_ood["accuracy"],
                  "f1 in-domain": r.row["f1"], "f1 out-of-domain": m_ood["f1"]})
pd.DataFrame(shift).round(4)

README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.78MB            

sentiment/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  901kB            

sentiment/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/validation-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  167kB            

... (3 outputs left out)

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12284 [00:00<?, ? examples/s]

OOD set: 1000 tweets, positive share = 0.377


,model,acc in-domain (IMDb),acc out-of-domain (tweets),f1 in-domain,f1 out-of-domain
0,DistilBERT (pretrained) / fine-tuning,0.8440,0.771,0.8468,0.7315
1,DistilBERT (pretrained) / frozen,0.8000,0.762,0.7957,0.7207
2,Transformer from scratch / from scratch,0.6875,0.463,0.6858,0.5491


In [10]:
fig, cm_ood = pc.plot_confusion(np.array(tw_labels), ood["fine-tuned"].argmax(1),
                                "confusion_matrix_ood.png", title="Confusion matrix (TweetEval, OOD)")
plt.show()

## Save results.csv

In [11]:
results = pd.DataFrame(rows)
results.to_csv("results.csv", index=False)
results[["experiment", "model", "mode", "train_frac", "max_len", "eval_set",
         "accuracy", "precision", "recall", "f1", "train_time_s", "peak_mem_mb"]].round(4)

,experiment,model,mode,train_frac,max_len,eval_set,accuracy,precision,recall,f1,train_time_s,peak_mem_mb
0,main,DistilBERT (pretrained),fine-tuning,1.00,128,IMDb test,0.8440,0.8320,0.8620,0.8468,88.1,2084
1,exp2_frozen_vs_ft,DistilBERT (pretrained),frozen,1.00,128,IMDb test,0.8000,0.8132,0.7790,0.7957,20.4,2521
2,exp1_scratch_vs_pretrained,Transformer from scratch,from scratch,1.00,128,IMDb test,0.6875,0.6896,0.6820,0.6858,7.0,908
3,exp3_data,DistilBERT (pretrained),fine-tuning,0.10,128,IMDb test,0.6995,0.6710,0.7830,0.7227,15.1,2893
4,exp3_data,Transformer from scratch,from scratch,0.10,128,IMDb test,0.5880,0.5926,0.5630,0.5774,1.0,1459
5,exp3_data,DistilBERT (pretrained),fine-tuning,0.25,128,IMDb test,0.8050,0.7955,0.8210,0.8081,27.1,2925
6,exp3_data,Transformer from scratch,from scratch,0.25,128,IMDb test,0.6265,0.6259,0.6290,0.6274,2.1,1461
7,exp3_data,DistilBERT (pretrained),fine-tuning,0.50,128,IMDb test,0.8430,0.8317,0.8600,0.8456,47.9,2925
8,exp3_data,Transformer from scratch,from scratch,0.50,128,IMDb test,0.6465,0.6386,0.6750,0.6563,3.5,1459
9,exp4_maxlen,DistilBERT (pretrained),fine-tuning,1.00,64,IMDb test,0.7920,0.7724,0.8280,0.7992,47.3,2263


## 21. Confidence analysis (fine-tuned model, IMDb test)

In [12]:
fig, conf_table = pc.plot_confidence(y_test, main.probs, "confidence_analysis.png")
plt.show()
print("IMDb test (in-domain)")
display(conf_table.round(4))
print("TweetEval (out-of-domain)")
display(pc.confidence_table(np.array(tw_labels), ood["fine-tuned"]).round(4))

IMDb test (in-domain)


,bin,n,share,accuracy,n_wrong
0,Low (0.5-0.7),285,0.1425,0.5018,142
1,Medium (0.7-0.9),463,0.2315,0.7581,112
2,High (>=0.9),1252,0.6260,0.9537,58


TweetEval (out-of-domain)


,bin,n,share,accuracy,n_wrong
0,Low (0.5-0.7),456,0.456,0.6294,169
1,Medium (0.7-0.9),442,0.442,0.8710,57
2,High (>=0.9),102,0.102,0.9706,3


## 20. Error analysis — examples (fine-tuned model, IMDb test)

In [13]:
errs = pc.pick_errors(data["test"][0], y_test, main.probs, n_fp=3, n_fn=3, n_conf=2)
table_md = pc.write_error_table(errs, "error_examples.md")
pd.set_option("display.max_colwidth", 300)
errs

,idx,selection,text,true,pred,confidence
0,1018,very confident & wrong,"I've been a fan of Jim Henson and his characters since the very beginning. The most beguiling thing about them was the love and innocence and camaraderie shown. Kermit was a role model of deep thinking and problem solving. A spiritual character, yet sweetly and believably so. All the other chara...",negative,positive,0.9828
1,833,very confident & wrong,"Each year the company called Nu produces couple of ""action packed"", ""full of suspense"" movies. This little nugget, called Shadowcaster III(Until I visited this site I wasn't avare this is a whole trilogy), is a great example of the good job the company is doing. Frank Zagarino is as mean as alwa...",negative,positive,0.9814
2,857,false positive,"Well, for a start, I must say that, here, in Russia, a saga of Geralt of Rivia is known and loved. Andrzej Sapkowski - a gifted writer, or, maybe, even genius of some sort - created one of the most realistic, honest, cynical and God d@mn well written fantasy worlds in a history of literature. An...",negative,positive,0.9807
3,874,false positive,"Earnest effort which achieves some success to adapt the classic Odyssey story to a '30's nostalgia period piece. The adventurers this time are escaped convicts, wandering about the Depression afflicted South bungling their way into trouble.<br /><br />The greatest strength of the film is the won...",negative,positive,0.9805
4,977,false positive,"If you never have read the book and never intend to read it in the future, go on and watch the movie (6/10). It is a nice fantasy movie with well done CGI, nice acting, a beautiful environment and an above-average fantasy story.<br /><br />If you have read the book like me about 10 times or more...",negative,positive,0.9767
5,694,false negative,"You have to be awfully patient to sit through a film with one-liners so flat and unfunny that you wonder what all the fuss was about when WHISTLING IN THE DARK opened to such an enthusiastic greeting from audiences in the 1940s.<br /><br />On top of some weak one-liners and ordinary sight gags, ...",positive,negative,0.9755
6,1129,false negative,"This is not the kind of movie that really merits critical attention. It's not going to win any Oscars - it's really not a very good movie at all. Heck, it's not even really a movie - just a string of short disgusting, gross-out skits strung together in an effort to make entertainment. And as muc...",positive,negative,0.9734
7,795,false negative,"Every episode I saw when I was an innocent child was stupid. There were some funny looking puppets called ""The Mitts"" who would always discuss childhood things. They would play this song before ""The Mitts"" part of the show that goes like this ""Let's join the Mitts..."" There was a Groucho Marx pu...",positive,negative,0.9726


In [14]:
# predictions for later inspection
pd.DataFrame({"text": data["test"][0], "true": y_test, "pred": pred_main,
              "p_positive": main.probs[:, 1], "confidence": main.probs.max(1)}
             ).to_csv("predictions_test.csv", index=False)

## 23. Reproducibility

In [15]:
repro = {
    "Model": cfg.model_name,
    "Tokenizer": f"{cfg.model_name} (WordPiece, uncased)",
    "Dataset": "stanfordnlp/imdb (OOD: cardiffnlp/tweet_eval sentiment, neutral removed)",
    "Train split": f"{cfg.n_train} reviews from IMDb train (shuffled, seed {cfg.seed})",
    "Validation split": f"{cfg.n_val} reviews from IMDb train (disjoint from train)",
    "Test split": f"{cfg.n_test} reviews from IMDb test (shuffled, seed {cfg.seed})",
    "Max sequence length": f"{cfg.max_len} (Exp 4: 64/128/256)",
    "Batch size": cfg.batch_size,
    "Learning rate": f"fine-tune {cfg.ft_lr}, frozen head {cfg.frozen_lr}, scratch {cfg.scratch_lr}",
    "Epochs": f"fine-tune {cfg.ft_epochs}, frozen head {cfg.frozen_epochs}, scratch {cfg.scratch_epochs}",
    "Optimizer": f"AdamW (weight decay {cfg.weight_decay}), linear schedule, warmup {cfg.warmup_ratio:.0%}",
    "Random seed": cfg.seed,
    "Hardware": pc.hardware_info(),
    "Training time": f"main fine-tune {main.row['train_time_s']} s; whole notebook {time.perf_counter() - T_START:.0f} s",
    "Library versions": f"torch {torch.__version__}, transformers {__import__('transformers').__version__}, datasets {__import__('datasets').__version__}",
}
with open("reproducibility.json", "w", encoding="utf-8") as f:
    json.dump(repro, f, indent=2, ensure_ascii=False)
for k, v in repro.items():
    print(f"{k}: {v}")

Model: distilbert-base-uncased
Tokenizer: distilbert-base-uncased (WordPiece, uncased)
Dataset: stanfordnlp/imdb (OOD: cardiffnlp/tweet_eval sentiment, neutral removed)
Train split: 4000 reviews from IMDb train (shuffled, seed 42)
Validation split: 1000 reviews from IMDb train (disjoint from train)
Test split: 2000 reviews from IMDb test (shuffled, seed 42)
Max sequence length: 128 (Exp 4: 64/128/256)
Batch size: 32
Learning rate: fine-tune 2e-05, frozen head 0.001, scratch 0.0005
Epochs: fine-tune 2, frozen head 30, scratch 5
Optimizer: AdamW (weight decay 0.01), linear schedule, warmup 10%
Random seed: 42
Hardware: x86_64, 2 logical cores, 13 GB RAM, Tesla T4
Training time: main fine-tune 88.1 s; whole notebook 602 s
Library versions: torch 2.14.1+cu130, transformers 5.22.1, datasets 5.2.0


## Download outputs (Colab)

In [16]:
import zipfile, glob
outs = ["results.csv", "data_efficiency.png", "confusion_matrix.png", "confusion_matrix_ood.png",
        "confidence_analysis.png", "error_examples.md", "predictions_test.csv", "reproducibility.json"]
with zipfile.ZipFile("lab08_outputs.zip", "w") as z:
    for f in outs:
        z.write(f)
print("saved lab08_outputs.zip:", outs)
try:
    from google.colab import files
    files.download("lab08_outputs.zip")
except Exception as e:  # not on Colab, or no browser frontend (e.g. nbconvert)
    print("download skipped:", type(e).__name__)

saved lab08_outputs.zip: ['results.csv', 'data_efficiency.png', 'confusion_matrix.png', 'confusion_matrix_ood.png', 'confidence_analysis.png', 'error_examples.md', 'predictions_test.csv', 'reproducibility.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>